## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
import json


## 2. Load Datset

In [2]:
df = pd.read_csv("data.csv")

print("Shape:", df.shape)
df.head(2)

Shape: (3044, 10)


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN


In [3]:
df.columns = [
    "sr_no",
    "date",
    "startup_name",
    "industry_vertical",
    "subvertical",
    "city",
    "investors",
    "investment_type",
    "amount_usd",
    "remarks"
]

print("Renamed columns:", df.columns.tolist())

Renamed columns: ['sr_no', 'date', 'startup_name', 'industry_vertical', 'subvertical', 'city', 'investors', 'investment_type', 'amount_usd', 'remarks']


## 4. EDA

In [4]:
print(df.isnull().sum())

sr_no                   0
date                    0
startup_name            0
industry_vertical     171
subvertical           936
city                  180
investors              24
investment_type         4
amount_usd            960
remarks              2625
dtype: int64


In [5]:
print("Duplicates:", df.duplicated().sum()) # check for dubplicates 


Duplicates: 0


In [6]:
print(df["amount_usd"].dropna().unique()[:20]) # Check for any undeifined or wrong entery

['20,00,00,000' '80,48,394' '1,83,58,860' '30,00,000' '18,00,000'
 '90,00,000' '15,00,00,000' '60,00,000' '7,00,00,000' '5,00,00,000'
 '2,00,00,000' '1,20,00,000' '3,00,00,000' '59,00,000' '20,00,000'
 '23,10,00,000' '4,86,000' '15,00,000' 'undisclosed' '2,60,00,000']


In [7]:
print(df["city"].value_counts().head(20)) # Check for top cities

city
Bangalore     700
Mumbai        567
New Delhi     421
Gurgaon       287
Bengaluru     141
Pune          105
Hyderabad      99
Chennai        97
Noida          92
Gurugram       50
Ahmedabad      38
Delhi          34
Jaipur         30
Kolkata        21
Indore         13
Chandigarh     11
Goa            10
Vadodara       10
Singapore       8
Coimbatore      5
Name: count, dtype: int64


In [8]:
def clean_startup_name(name):
    name = str(name).strip()

    if name.lower() in ["", "nan", "none"]:
        return None

    if name.startswith("http"):
        match = re.search(r"https?://(?:www\.)?([^/]+)", name)
        return match.group(1).lower() if match else None

    name = name.lower().strip()
    name = name.strip('"').strip("'")                      
    name = name.replace("\\'", "'").replace('\\"', '"').replace("\\\\", "")
    return name

df["startup_name"] = df["startup_name"].apply(clean_startup_name)

In [9]:
df["date"] = pd.to_datetime(df["date"], errors="coerce") # converting date to a one proper formate
df["year"] = df["date"].dt.year


In [10]:
df["amount_usd"] = (
    df["amount_usd"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(",", "", regex=False)      # Removing comma from amount value
    .replace("undisclosed", np.nan)         # convert text values to nan so that math oprations can be performed later 
    .replace("nan",  np.nan)
    .replace("none", np.nan)
)
df["amount_usd"] = pd.to_numeric(df["amount_usd"], errors="coerce") # converting whole column to numeric


print(df["amount_usd"].head(10))

0    200000000.0
1      8048394.0
2     18358860.0
3      3000000.0
4      1800000.0
5      9000000.0
6    150000000.0
7      6000000.0
8     70000000.0
9     50000000.0
Name: amount_usd, dtype: float64


In [11]:
city_map = {
    "bengaluru":  "bangalore",
    "bengalore":  "bangalore",
    "banglore":   "bangalore",
    "new delhi":  "delhi",
    "bombay":     "mumbai",
    "gurugram":   "gurgaon",   
}

def clean_city(val):
    if pd.isna(val) or str(val).strip().lower() in ["", "nan", "none"]:
        return None
    val = str(val).lower().strip()
    val = re.sub(r"[^a-z /]", "", val).strip()

    if "/" in val:
        val = val.split("/")[0].strip()

    return city_map.get(val, val)

df["city"] = df["city"].apply(clean_city)

print("Top cities after normalization:")
print(df["city"].value_counts().head())

Top cities after normalization:
city
bangalore    848
mumbai       572
delhi        459
gurgaon      338
pune         112
Name: count, dtype: int64


In [12]:
def clean_text(val):
    """ A small function to clean text and making it consistent"""
    
    if pd.isna(val) or str(val).strip().lower() in ["", "nan", "none"]:
        return None

    val = str(val).encode("utf-8", "ignore").decode("utf-8", "ignore")

    val = re.sub(r"[^a-zA-Z0-9 &\-\/,.()]", " ", val)
    val = re.sub(r"\s+", " ", val).lower().strip()
    return val if val else None

TEXT_COLS = ["industry_vertical", "subvertical", "investors", "investment_type", "remarks"]
for col in TEXT_COLS:
    df[col] = df[col].apply(clean_text) 


df = df.where(pd.notnull(df), None) # keep all null representation same 

print(df.isnull().sum())

sr_no                   0
date                 1752
startup_name            0
industry_vertical     171
subvertical           936
city                  180
investors              24
investment_type         4
amount_usd            979
remarks              2625
year                 1752
dtype: int64


In [13]:
# Check for all top industry present 
print(f"Unique industry: {df['industry_vertical'].nunique()}")
print("\nTop 30:")
print(df["industry_vertical"].value_counts().head(10))

Unique industry: 799

Top 30:
industry_vertical
consumer internet    942
technology           478
ecommerce            258
healthcare            71
finance               62
e-commerce            41
logistics             32
education             24
food & beverage       23
ed-tech               15
Name: count, dtype: int64


In [14]:
def normalize_industry(value):
    """ clubbing similar industry to one main category"""

    if not value or str(value).strip().lower() in ["none", "others", "other", "nan", ""]:
        return "other"
    v = str(value).lower().strip()

    if any(k in v for k in ["fin", "payment", "lending", "nbfc", "insurance", "wealth", "invest"]):
        return "fintech"
    if any(k in v for k in ["edu", "e-tech", "e-learn", "learning", "coaching", "exam", "skill"]):
        return "edtech"
    if any(k in v for k in ["food", "beverage", "restaurant", "dining", "meal", "grocer"]):
        return "food"
    if any(k in v for k in ["logistics", "delivery", "freight", "warehou", "supply chain"]):
        return "logistics"
    if any(k in v for k in ["health", "medical", "pharma", "diagnostic", "clinical", "doctor"]):
        return "healthcare"
    if any(k in v for k in ["ecommerce", "e-commerce", "consumer internet", "retail", "marketplace", "etailer"]):
        return "ecommerce"
    if any(k in v for k in ["real estate", "property", "realty", "housing", "rental"]):
        return "real_estate"
    if any(k in v for k in ["auto", "transport", "cab", "ride", "mobility", "bike", "vehicle"]):
        return "mobility"
    if any(k in v for k in ["social", "media", "content", "entertainment", "gaming", "video"]):
        return "media_entertainment"
    if "saas" in v:
        return "saas"
    if "fmcg" in v or "consumer goods" in v:
        return "fmcg"
    if any(k in v for k in ["tech", "software", "cloud", "ai", "data", "iot", "it", "information technology", "technology"]):
        return "technology"

    return "other"

df["industry"] = df["industry_vertical"].apply(normalize_industry)

print("industry clean")
print(df["industry"].head(10))

industry clean
0        edtech
1      mobility
2     ecommerce
3       fintech
4         other
5     logistics
6    technology
7    technology
8     ecommerce
9         other
Name: industry, dtype: object


In [15]:
def row_to_natural_text(row):
    """
    convert the normal rows to a natrual sentance with some helping text.
    """
    parts = []

    if row["startup_name"]:
        parts.append(f"Startup Name: {row['startup_name']}.")

    if row["industry"]:
        parts.append(f"Sector: {row['industry']}.")

    if row["industry_vertical"]:
        parts.append(f"Industry: {row['industry_vertical']}.")

    if row["subvertical"]:
        parts.append(f"Sub-sector: {row['subvertical']}.")

    if row["city"]:
        parts.append(f"Located in {row['city']}, India.")

    if row["investment_type"]:
        parts.append(f"Investment type: {row['investment_type']}.")

    if pd.notna(row["amount_usd"]) and row["amount_usd"] is not None:
        parts.append(f"Amount raised: ${row['amount_usd']} USD.")

    if row["investors"]:
        parts.append(f"Investors: {row['investors']}.")

    if pd.notna(row["date"]):
        parts.append(f"Funding date: {row['date'].strftime('%B %Y')}.")

    if row["remarks"]:
        parts.append(f"Remarks: {row['remarks']}.")

    return " ".join(parts)

df["embedding_text"] = df.apply(row_to_natural_text, axis=1)


print(df["embedding_text"].iloc[35]) # check if its created properly 

Startup Name: fpl technologies. Sector: fintech. Industry: fintech. Sub-sector: financial services. Located in pune, India. Investment type: maiden round. Amount raised: $4500000.0 USD. Investors: matrix partners india, sequoia india. Funding date: May 2019.


The dataset has one row per funding round but user can ask for total as well as for separate funding value so we create a new small dataset here.

If we just sum amount_usd per company, we lose the ability to answer:
- "How much did Razorpay raise in Series B specifically?"
- "Who were the seed round investors for Swiggy?"


In [16]:
def build_funding_rounds(group):
    rounds = []
    for _, row in group.iterrows():
        rounds.append({
            "round_type": row["investment_type"] or None,
            "amount_usd": float(row["amount_usd"]) if pd.notna(row["amount_usd"]) else None,
            "investors":  row["investors"] or None,
            "date":       row["date"].strftime("%Y-%m-%d") if pd.notna(row["date"]) else None
        })
    return rounds

funding_rounds_map = (
    df.groupby("startup_name")
      .apply(build_funding_rounds) 
      .to_dict()
)


print("Swiggy funding rounds:")
for r in funding_rounds_map.get("swiggy", []):
    print(r)

Swiggy funding rounds:
{'round_type': 'private equity', 'amount_usd': 100000000.0, 'investors': 'naspers', 'date': '2018-07-02'}
{'round_type': 'private equity', 'amount_usd': 80000000.0, 'investors': 'nasper, accel india, saif partners, bessemer venture partners, harmony partners, norwest venture partners', 'date': None}
{'round_type': 'private equity', 'amount_usd': 15000000.0, 'investors': 'bessemer venture partners', 'date': None}
{'round_type': 'private equity', 'amount_usd': 7000000.0, 'investors': 'norwest venture partners, dst global, accel partners', 'date': '2016-10-05'}
{'round_type': 'private equity', 'amount_usd': 35000000.0, 'investors': 'harmony partners, rb investments & existing investors', 'date': None}
{'round_type': 'private equity', 'amount_usd': 16500000.0, 'investors': 'norwest venture partners, saif partners, accel partners', 'date': '2015-09-06'}
{'round_type': 'private equity', 'amount_usd': 15000000.0, 'investors': 'norwest venture partners', 'date': '2015-06

C:\Users\91845\AppData\Local\Temp\ipykernel_18228\1447151909.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_funding_rounds)


Aggregates all rounds into a single company profile why?
Because user can ask for all funding info at once as well

In [17]:
def agg_unique_list(series):
    """For non-investor fields — each value is one complete item."""
    seen, result = set(), []
    for v in series.dropna():
        v = str(v).strip()
        if v and v.lower() not in ("nan", "none", "") and v not in seen:
            seen.add(v)
            result.append(v)
    return result

def agg_investors_list(series):
    """
    For investors field only — splits comma-separated strings into individual names.
    "tiger global, y combinator" → ["tiger global", "y combinator"]
    Also filters out generic placeholders like "undisclosed", "not disclosed".
    """
    SKIP = ("nan", "none", "", "undisclosed", "not disclosed", 
            "group of investors", "angel investors")
    seen, result = set(), []
    for v in series.dropna():
        for part in str(v).split(","):
            part = part.strip()
            if part and part.lower() not in SKIP and part not in seen:
                seen.add(part)
                result.append(part)
    return result


company_df = df.groupby("startup_name").agg(
    industry          = ("industry",          "first"),
    industry_vertical = ("industry_vertical", agg_unique_list),
    subverticals      = ("subvertical",       agg_unique_list),
    city              = ("city",              "first"),
    all_cities        = ("city",              agg_unique_list),
    num_rounds        = ("investment_type",   "count"),
    all_investors     = ("investors",         agg_investors_list),  # ← uses split version
    first_funded      = ("date",              "min"),
    last_funded       = ("date",              "max"),
    all_remarks       = ("remarks",           agg_unique_list),
).reset_index()

# Attach full per-round breakdown
company_df["funding_rounds"] = company_df["startup_name"].map(funding_rounds_map)

# Derive total_funding from disclosed round amounts only
def calc_total_funding(rounds):
    amounts = [r["amount_usd"] for r in rounds if r["amount_usd"] is not None]
    return sum(amounts) if amounts else None

company_df["total_funding"] = company_df["funding_rounds"].apply(calc_total_funding)

print(f"Unique companies: {len(company_df)}")
print("\nSample:")
print(company_df[["startup_name", "industry", "city", "num_rounds", "total_funding"]].head(5))

# Verify investor split worked
razorpay = company_df[company_df["startup_name"] == "razorpay"]
if len(razorpay):
    for inv in razorpay.iloc[0]["all_investors"]:
        print(f"  '{inv}'")

Unique companies: 2349

Sample:
  startup_name    industry     city  num_rounds  total_funding
0        #fame       other     None           1     10000000.0
1    121policy   ecommerce  kolkata           1            NaN
2    19th mile  technology  gurgaon           1       180000.0
3       1crowd     fintech   mumbai           1            NaN
4          1mg   ecommerce  gurgaon           4     40000000.0
  'tiger global'
  'y combinator'
  'mastercard'
  'tiger global management'
  'matrix partners'
  'punit soni'


In [18]:
# Verification — check a well-known multi-round company
for name in ["swiggy","paytm"]:
    row = company_df[company_df["startup_name"] == name]
    if len(row):
        r = row.iloc[0]
        print(f"\n--- {name} ---")
        print(f"  num_rounds   : {r['num_rounds']}")
        total = f"${r['total_funding']:,.0f}" if r["total_funding"] else "undisclosed"
        print(f"  total_funding: {total}")
        print(f"  all_investors: {r['all_investors'][:5]} ...")
        print("  funding_rounds breakdown:")
        for rd in r["funding_rounds"]:
            amt = f"${rd['amount_usd']:,.0f}" if rd["amount_usd"] else "undisclosed"
            print(f"    [{rd['date']}] {str(rd['round_type']):28s} {amt:>18}  {(rd['investors'] or 'N/A')[:35]}")


--- swiggy ---
  num_rounds   : 8
  total_funding: $270,500,000
  all_investors: ['naspers', 'nasper', 'accel india', 'saif partners', 'bessemer venture partners'] ...
  funding_rounds breakdown:
    [2018-07-02] private equity                     $100,000,000  naspers
    [None] private equity                      $80,000,000  nasper, accel india, saif partners,
    [None] private equity                      $15,000,000  bessemer venture partners
    [2016-10-05] private equity                       $7,000,000  norwest venture partners, dst globa
    [None] private equity                      $35,000,000  harmony partners, rb investments & 
    [2015-09-06] private equity                      $16,500,000  norwest venture partners, saif part
    [2015-06-05] private equity                      $15,000,000  norwest venture partners
    [2015-03-04] private equity                       $2,000,000  accel partners, saif partners

--- paytm ---
  num_rounds   : 7
  total_funding: $3,148,95

Building one text paragraph per company — which will be used to build KB for RAG
Includes per-round breakdown so round-specific questions can be answered

In [19]:
import math

def company_to_text(row):

    parts = []

    # Identity
    industry = row["industry"] or "unknown sector"
    article  = "an" if industry[0].lower() in "aeiou" else "a"


    if row["city"]:
        parts.append(f"{row['startup_name']} is {article} {industry} startup based in {row['city']}, India.")
    else:
        parts.append(f"{row['startup_name']} is {article} {industry} startup based in India.")

    # Industry verticals
    if row["industry_vertical"]:
        parts.append(f"Industry vertical: {', '.join(row['industry_vertical'][:3])}.")

    # Sub-sectors
    if row["subverticals"]:
        parts.append(f"Sub-sectors: {', '.join(row['subverticals'][:3])}.")

    # Multi-city presence
    extra_cities = [c for c in (row["all_cities"] or []) if c != row["city"]]
    if extra_cities:
        parts.append(f"Also has presence in: {', '.join(extra_cities[:3])}.")

    # Per-round breakdown
    if row["funding_rounds"]:
        round_lines = []
        for r in row["funding_rounds"]:
            rtype = r["round_type"] or "undisclosed round"
            amt   = f"${r['amount_usd']:,.0f} USD" if r["amount_usd"] else "undisclosed amount"
            inv   = r["investors"] or "undisclosed investors"
            date  = r["date"][:7] if r["date"] else ""
            round_lines.append(f"{rtype}: {amt} from {inv} ({date})")
        parts.append(
            f"Funding rounds ({len(row['funding_rounds'])} total): "
            + "; ".join(round_lines) + "."
        )

    # Total disclosed funding
    total = row["total_funding"]
    if total is not None and not (isinstance(total, float) and math.isnan(total)):
        parts.append(f"Total disclosed funding: ${total:,.0f} USD.")
    else:
        parts.append("Total funding amount undisclosed.")

    # Timeline
    first = row["first_funded"].strftime("%B %Y") if pd.notna(row["first_funded"]) else None
    last  = row["last_funded"].strftime("%B %Y")  if pd.notna(row["last_funded"])  else None
    if first and last and first != last:
        parts.append(f"First funded in {first}, most recently in {last}.")
    elif first:
        parts.append(f"Funded in {first}.")

    # All unique investors
    if row["all_investors"]:
        investors = ", ".join(row["all_investors"][:20])
        overflow  = len(row["all_investors"]) - 20
        suffix    = f" and {overflow} more" if overflow > 0 else ""
        parts.append(f"All investors: {investors}{suffix}.")

    # Remarks
    if row["all_remarks"]:
        parts.append(f"Notes: {'; '.join(row['all_remarks'][:2])}.")

    return " ".join(parts)


company_df["embedding_text"] = company_df.apply(company_to_text, axis=1)

# Spot-check
for name in ["swiggy", "razorpay", "#fame"]:  # added #fame — no city, tests the fix
    row = company_df[company_df["startup_name"] == name]
    if len(row):
        print(f"\n--- {name} ---")
        print(row.iloc[0]["embedding_text"])


--- swiggy ---
swiggy is a food startup based in bangalore, India. Industry vertical: food and beverages, consumer internet, online food ordering. Sub-sectors: online food delivery, online food delivery platform, online food ordering & delivery. Funding rounds (8 total): private equity: $100,000,000 USD from naspers (2018-07); private equity: $80,000,000 USD from nasper, accel india, saif partners, bessemer venture partners, harmony partners, norwest venture partners (); private equity: $15,000,000 USD from bessemer venture partners (); private equity: $7,000,000 USD from norwest venture partners, dst global, accel partners (2016-10); private equity: $35,000,000 USD from harmony partners, rb investments & existing investors (); private equity: $16,500,000 USD from norwest venture partners, saif partners, accel partners (2015-09); private equity: $15,000,000 USD from norwest venture partners (2015-06); private equity: $2,000,000 USD from accel partners, saif partners (2015-03). Total d

In [20]:
print("\n=== company_df (aggregated) ===")
print(f"Companies         : {len(company_df)}")
print(f"Null embedding_text: {company_df['embedding_text'].isna().sum()}")
print(f"industry dist:\n{company_df['industry'].value_counts()}")


=== company_df (aggregated) ===
Companies         : 2349
Null embedding_text: 0
industry dist:
industry
ecommerce              1019
technology              537
other                   310
fintech                  99
healthcare               96
food                     85
logistics                57
edtech                   52
media_entertainment      32
mobility                 28
real_estate              23
saas                      6
fmcg                      5
Name: count, dtype: int64


In [21]:
len(df["embedding_text"].unique())

3042

In [23]:
COLS_TO_DROP = ["embedding_text", "amount_disclosed","industry_vertical"]

export_df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns]).copy()
export_df["date"] = export_df["date"].astype(str)

export_df.to_csv("startup_funding_clean.csv", index=False)
print(f"startup_funding_clean.csv  →  {len(export_df):,} rows  |  {export_df.shape[1]} columns")
print(f"  Columns: {export_df.columns.tolist()}")

startup_funding_clean.csv  →  3,044 rows  |  11 columns
  Columns: ['sr_no', 'date', 'startup_name', 'subvertical', 'city', 'investors', 'investment_type', 'amount_usd', 'remarks', 'year', 'industry']


In [24]:
import math

def df_to_json(obj):
    """
    Recursively walk the object and replace anything
    that is not valid JSON with None (serializes as null).
    Handles: float NaN, pandas NaT, string "NaT", string "nan"
    """
    if obj is None:
        return None
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, str) and obj.lower() in ("nat", "nan", "none", ""):
        return None
    if isinstance(obj, dict):
        return {k: df_to_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [df_to_json(i) for i in obj]
    return obj

company_export = company_df.copy()

company_export["first_funded"] = company_export["first_funded"].astype(str)
company_export["last_funded"]  = company_export["last_funded"].astype(str)

records = company_export.to_dict(orient="records")
records = [df_to_json(r) for r in records]

# Verify no NaN/NaT slipped through
raw = json.dumps(records)
assert "NaN"   not in raw, "NaN found in output!"
assert '"NaT"' not in raw, "NaT string found in output!"

with open("company_profiles.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print(f"company_profiles.json → {len(records):,} companies")
print(f"Keys per company: {list(records[0].keys())}")


# Spot checks
sample = next(c for c in records if c["startup_name"] == "swiggy")
print("\nswiggy:")
print("  total_funding :", sample["total_funding"])   
print("  first_funded  :", sample["first_funded"])     
print("  embedding_text:", sample["embedding_text"][:1000])  

company_profiles.json → 2,349 companies
Keys per company: ['startup_name', 'industry', 'industry_vertical', 'subverticals', 'city', 'all_cities', 'num_rounds', 'all_investors', 'first_funded', 'last_funded', 'all_remarks', 'funding_rounds', 'total_funding', 'embedding_text']

swiggy:
  total_funding : 270500000.0
  first_funded  : 2015-03-04
  embedding_text: swiggy is a food startup based in bangalore, India. Industry vertical: food and beverages, consumer internet, online food ordering. Sub-sectors: online food delivery, online food delivery platform, online food ordering & delivery. Funding rounds (8 total): private equity: $100,000,000 USD from naspers (2018-07); private equity: $80,000,000 USD from nasper, accel india, saif partners, bessemer venture partners, harmony partners, norwest venture partners (); private equity: $15,000,000 USD from bessemer venture partners (); private equity: $7,000,000 USD from norwest venture partners, dst global, accel partners (2016-10); private eq